In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
import random
import yaml

In [ ]:
REPO_ROOT = "/content/drive/MyDrive/RutishauserLab/AI4Sci/METHODS/DiffusionBANMap/"
CONFIG_FILE = "config/betavae_full_celeba.yaml"
CKPT_PATH = None # None → load latest checkpoint from experiment output dir
N_SAMPLES = 8 # images to show in reconstruction grid
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
sys.path.insert(0, REPO_ROOT)

import importlib.metadata
if not hasattr(importlib.metadata, '_patched'):
    _original_version = importlib.metadata.version
    def _patched_version(name):
        try:
            return _original_version(name)
        except importlib.metadata.PackageNotFoundError:
            return 'unknown'
    importlib.metadata.version = _patched_version
    importlib.metadata._patched = True

from data_utils import build_dataloaders
from models.beta_vae import BetaVAE

## Load model & checkpoint

In [ ]:
REPO_ROOT = os.path.abspath("..")

with open(os.path.join(REPO_ROOT, CONFIG_PATH)) as f:
    cfg = types.SimpleNamespace(**yaml.safe_load(f))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BetaVAE(
    input_channels=cfg.input_channels,
    input_height=cfg.input_height,
    input_width=cfg.input_width,
    latent_dim=cfg.latent_dim,
    hidden_dim=cfg.hidden_dim,
    beta=cfg.beta,
    device=device,
)

if CKPT_PATH is None:
    ckpt_dir = os.path.join(REPO_ROOT, cfg.output_dir, "checkpoints")
    ckpts = glob.glob(os.path.join(ckpt_dir, "*.pt"))
    if not ckpts:
        raise FileNotFoundError(f"No checkpoints found in {ckpt_dir}")
    CKPT_PATH = max(ckpts, key=lambda p: int(p.rsplit("step", 1)[-1].replace(".pt", "")))

ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Loaded: {CKPT_PATH}  (step {ckpt['step']})")

## Reconstructions

In [ ]:
_, val_dl = build_dataloaders(cfg)
all_images = torch.cat([b for b in val_dl], dim=0)

indices = random.sample(range(len(all_images)), min(N_SAMPLES, len(all_images)))
originals = all_images[indices].to(device)

with torch.no_grad():
    x_recon, _, _ = model(originals)
    x_recon = torch.sigmoid(x_recon).cpu()

originals_cpu = originals.cpu()
n = len(indices)

fig, axes = plt.subplots(2, n, figsize=(n * 2, 5))
fig.suptitle("Top: original  |  Bottom: beta-VAE reconstruction", fontsize=11)
for i in range(n):
    axes[0, i].imshow(originals_cpu[i].permute(1, 2, 0))
    axes[0, i].axis("off")
    axes[1, i].imshow(x_recon[i].permute(1, 2, 0))
    axes[1, i].axis("off")
plt.tight_layout()
plt.show()

## Latent activation distributions

Pass every image in the dataset through the encoder and collect the posterior mean μ for each latent dimension.

In [ ]:
all_mu = []
with torch.no_grad():
    for batch in val_dl:
        batch = batch.to(device)
        _, mu, _ = model(batch)
        all_mu.append(mu.cpu())

all_mu = torch.cat(all_mu, dim=0).numpy()  # [N, latent_dim]
print(f"Collected μ for {all_mu.shape[0]} images across {all_mu.shape[1]} latent dimensions")